# M5-SV Version190 — Publication-only repair
CPU is sufficient. Enable Internet and the existing GITHUB_TOKEN and KAGGLE_API_TOKEN secrets. This entry recovers metadata from bingjiangli/minicells Version190, verifies its original source identity, and publishes without running tests, model forwards or optimizer updates. The original experiment status/error is preserved.

In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time
started=time.monotonic()
hard_deadline=started+3600
repo=Path('/kaggle/working/minicells-epclm004-m5-sv-publication-repair')
branch='research/epclm-004-m5-scientific-validation'

def run(command,cwd=None,limit=180,stop_at=None,log=None):
    remaining=min(limit,(stop_at or hard_deadline)-time.monotonic())
    if remaining<=0:raise TimeoutError('M5-SV budget exhausted; Output retained')
    stream=open(log,'a',buffering=1) if log else None
    try:
        p=subprocess.Popen(command,cwd=cwd,start_new_session=True,
                           stdout=stream,stderr=subprocess.STDOUT if stream else None)
        try:code=p.wait(timeout=remaining)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid,signal.SIGTERM)
            try:p.wait(timeout=15)
            except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()
            raise TimeoutError('M5-SV stopped at budget boundary; inspect retained Output')
    finally:
        if stream:stream.close()
    if log:print(Path(log).read_text(),flush=True)
    if code:raise RuntimeError('M5-SV stopped; inspect decision.json and raw.log in Output')

if not repo.exists():
    run(['git','clone','--depth=1','--single-branch','--branch',branch,'https://github.com/ArcheLabs/mini-cells.git',str(repo)])
else:
    origin=subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()
    if origin.removesuffix('.git')!='https://github.com/ArcheLabs/mini-cells':raise ValueError('unrelated checkout')
    run(['git','-C',str(repo),'fetch','--depth=1','origin',branch])
    run(['git','-C',str(repo),'switch','--detach','FETCH_HEAD'])
source=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
base=repo/'research/projects/cellular-language-models/experiments/epclm-004-unified-memory-10m/m5-sv'
out=Path('/kaggle/working/epclm004-m5-sv-publication-repair-v190');out.mkdir(exist_ok=True)
run([sys.executable,'-m','pip','install','-q','einops==0.8.2'],limit=300)
os.environ['OMP_NUM_THREADS']='1';os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD']='1'
# Check the exact publication module before spending any experiment budget.
run([sys.executable,str(base/'src/sv_publication.py')],cwd=repo,limit=60)
# Republish the exact failed Output; never invoke the experiment launcher.
run([sys.executable,str(base/'repair_publication.py'),'--destination',str(out)],
    cwd=repo,limit=3300,stop_at=started+3300,log=out/'repair.log')
